# Advanced Data Mining for Data-Driven Insights and Predictive Modeling
## Deliverable 1: Data Collection, Cleaning, and Exploration

**Dataset:** UCI Online Retail  
**Source:** UCI Machine Learning Repository  
**Dataset period:** 1 December 2010 – 9 December 2011

### Project purpose
This notebook loads, inspects, cleans, and explores the Online Retail transaction dataset. The analysis prepares a reliable foundation for later regression, classification, clustering, and association-rule mining.

### Research questions
1. What data-quality issues must be addressed before analysis?
2. How do sales vary over time and across countries?
3. Which products contribute most to sales?
4. What unusual values or outliers appear in quantity and unit price?
5. How can the findings guide later predictive and pattern-mining tasks?

**Important:** The workbook `Online Retail.xlsx` should be in the same folder as this notebook. The cleaning rules below preserve the raw dataset and create separate analysis datasets so that returns/cancellations are not silently discarded.


## 1. Import libraries and load the dataset

Pandas is used for data manipulation, NumPy for numerical operations, and Matplotlib/Seaborn for visualization. If a library is missing in your environment, uncomment and run the installation command below.

```python
# %pip install pandas numpy matplotlib seaborn openpyxl
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set a consistent visual style for all charts.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

# Load the Excel workbook from the same folder as this notebook.
file_path = Path("Online Retail.xlsx")
if not file_path.exists():
    raise FileNotFoundError(
        "Online Retail.xlsx was not found. Place the Excel file in the same folder as this notebook."
    )

raw_df = pd.read_excel(file_path, engine="openpyxl")
print(f"Dataset loaded: {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns")
display(raw_df.head())


## 2. Dataset description and suitability

The UCI Online Retail dataset contains transaction-level records from a UK-based online retailer. Its eight attributes describe invoice identifiers, product identifiers and descriptions, quantities, invoice dates, unit prices, customer identifiers, and countries.

The dataset is appropriate because it contains more than 500 records and includes numerical, categorical, date/time, customer, and product information. These features can support sales analysis and later predictive modeling. Transaction-level product records can also be grouped into invoices for market-basket/association-rule analysis.

**Dataset selection note:** The assignment asks for 8–10 or more attributes. This dataset has exactly 8 attributes. It meets the stated minimum of 8 attributes, but if the instructor interprets the requirement as strictly 10 attributes, confirm acceptability before final submission.


## 3. Inspect the structure and data types

In [ ]:
# Inspect dimensions, column names, data types, and memory usage.
print("Shape:", raw_df.shape)
print("\nColumn names:")
print(raw_df.columns.tolist())
print("\nData types:")
display(raw_df.dtypes.to_frame("Data type"))
print("\nDataset information:")
raw_df.info()


## 4. Assess data quality

In [ ]:
# Count missing values and calculate their percentages.
missing_summary = pd.DataFrame({
    "Missing count": raw_df.isna().sum(),
    "Missing percent": (raw_df.isna().mean() * 100).round(2)
}).sort_values("Missing count", ascending=False)
display(missing_summary)

# Count exact duplicate rows.
duplicate_count = raw_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count:,}")

# Identify unusual transaction values.
quality_checks = pd.Series({
    "Negative quantities": (raw_df["Quantity"] < 0).sum(),
    "Zero quantities": (raw_df["Quantity"] == 0).sum(),
    "Negative unit prices": (raw_df["UnitPrice"] < 0).sum(),
    "Zero unit prices": (raw_df["UnitPrice"] == 0).sum(),
    "Invoice numbers beginning with C": raw_df["InvoiceNo"].astype(str).str.startswith("C").sum()
})
display(quality_checks.to_frame("Record count"))


### Data-quality decisions

The initial inspection of this supplied workbook found:

- 541,909 rows and 8 columns.
- 135,080 missing `CustomerID` values.
- 1,454 missing `Description` values.
- 5,268 exact duplicate rows.
- 10,624 negative `Quantity` values.
- 2,515 zero `UnitPrice` values and 2 negative `UnitPrice` values.
- 9,288 invoice numbers beginning with `C`, which commonly indicate cancellations/credit transactions in this dataset.

**Cleaning approach and rationale**

1. Keep `raw_df` unchanged as the original source.
2. Remove exact duplicate rows in the analysis copy.
3. Convert text fields to consistent trimmed strings and parse dates.
4. Do not impute missing customer IDs: a fabricated customer identifier would create false customer-level relationships. Keep such rows for product/country analysis where appropriate, but exclude them from customer-level analysis.
5. Keep return/cancellation records identifiable in a separate transaction dataset. For ordinary positive-sales analysis, create a filtered dataset excluding cancelled invoices, non-positive quantities, and non-positive prices.
6. Do not automatically delete statistical outliers. Large orders may be genuine. Flag them and inspect their context; use robust summaries or log-scaled charts where appropriate.

These choices preserve information while making the analytical population explicit.


## 5. Clean and prepare analysis datasets

In [ ]:
# Work on a copy so the original imported data remains unchanged.
df = raw_df.copy()

# Standardize text columns by trimming leading/trailing whitespace.
for col in ["InvoiceNo", "StockCode", "Description", "Country"]:
    df[col] = df[col].astype("string").str.strip()

# Ensure dates are in datetime format and numeric columns are numeric.
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

# Remove exact duplicate rows.
before_dedup = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = before_dedup - len(df)

# Remove rows missing essential transaction fields.
essential_cols = ["InvoiceNo", "StockCode", "InvoiceDate", "Quantity", "UnitPrice", "Country"]
before_essential = len(df)
df = df.dropna(subset=essential_cols).copy()
essential_rows_removed = before_essential - len(df)

# Create a cancellation indicator before filtering sales.
df["IsCancellation"] = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

# Revenue at line-item level. Negative quantities will represent negative revenue/returns.
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Preserve all valid transaction lines (including returns/cancellations) for audit and return analysis.
transactions_all = df.copy()

# Positive-sales analysis population:
# - invoice is not marked as a cancellation
# - quantity is positive
# - unit price is positive
sales_df = df.loc[
    (~df["IsCancellation"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

# Customer-level work requires a known customer identifier.
customer_sales_df = sales_df.dropna(subset=["CustomerID"]).copy()

print(f"Exact duplicates removed: {duplicates_removed:,}")
print(f"Rows removed for missing essential transaction fields: {essential_rows_removed:,}")
print(f"All transaction lines after basic cleaning: {len(transactions_all):,}")
print(f"Positive-sales records for EDA: {len(sales_df):,}")
print(f"Positive-sales records with CustomerID: {len(customer_sales_df):,}")
print(f"Missing CustomerID retained in sales data: {sales_df['CustomerID'].isna().sum():,}")


### Why two cleaned datasets are retained

`transactions_all` retains valid return and cancellation records so that they can be studied separately and not confused with ordinary sales. `sales_df` is the positive-sales population used for sales trends, product performance, and most EDA. `customer_sales_df` is restricted to rows with a known customer ID for customer-level analysis.

The analysis uses line-item revenue calculated as `Quantity × UnitPrice`. This is a derived measure, not a column supplied in the original workbook.


## 6. Feature engineering and summary statistics

In [ ]:
# Add calendar features that may be useful in later analysis and modeling.
sales_df["Year"] = sales_df["InvoiceDate"].dt.year
sales_df["Month"] = sales_df["InvoiceDate"].dt.month
sales_df["YearMonth"] = sales_df["InvoiceDate"].dt.to_period("M").astype(str)
sales_df["DayOfWeek"] = sales_df["InvoiceDate"].dt.day_name()
sales_df["Hour"] = sales_df["InvoiceDate"].dt.hour

# Summarize the positive-sales dataset.
print("Positive-sales dataset shape:", sales_df.shape)
display(sales_df[["Quantity", "UnitPrice", "Revenue"]].describe().T)

# Overall sales metrics.
summary_metrics = pd.Series({
    "Number of invoices": sales_df["InvoiceNo"].nunique(),
    "Number of unique products": sales_df["StockCode"].nunique(),
    "Number of countries": sales_df["Country"].nunique(),
    "Number of identified customers": sales_df["CustomerID"].nunique(),
    "Total positive-sales revenue (£)": sales_df["Revenue"].sum(),
    "Average line-item revenue (£)": sales_df["Revenue"].mean()
})
display(summary_metrics.to_frame("Value"))


## 7. Exploratory Data Analysis (EDA)

### 7.1 Distribution of quantity, unit price, and revenue

In [ ]:
# These variables are highly skewed, so plots use a capped display range
# to make the main body of the distribution easier to inspect.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(sales_df.loc[sales_df["Quantity"] <= sales_df["Quantity"].quantile(0.99), "Quantity"],
             bins=40, ax=axes[0], color="steelblue")
axes[0].set_title("Quantity distribution (up to 99th percentile)")
axes[0].set_xlabel("Quantity")

sns.histplot(sales_df.loc[sales_df["UnitPrice"] <= sales_df["UnitPrice"].quantile(0.99), "UnitPrice"],
             bins=40, ax=axes[1], color="darkorange")
axes[1].set_title("Unit price distribution (up to 99th percentile)")
axes[1].set_xlabel("Unit price (£)")

sns.histplot(sales_df.loc[sales_df["Revenue"] <= sales_df["Revenue"].quantile(0.99), "Revenue"],
             bins=40, ax=axes[2], color="seagreen")
axes[2].set_title("Line-item revenue (up to 99th percentile)")
axes[2].set_xlabel("Revenue (£)")

plt.tight_layout()
plt.show()


**Interpretation prompt:** Describe whether each distribution is symmetric or skewed. Explain why the 99th-percentile display limit is used for readability, and clarify that the underlying records have not been deleted by this visualization step.


### 7.2 Identify potential outliers with boxplots

In [ ]:
# Boxplots are displayed on capped data ranges to prevent extreme values
# from compressing the central distribution.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

q99 = sales_df["Quantity"].quantile(0.99)
p99 = sales_df["UnitPrice"].quantile(0.99)

sns.boxplot(y=sales_df.loc[sales_df["Quantity"] <= q99, "Quantity"], ax=axes[0])
axes[0].set_title("Quantity boxplot (up to 99th percentile)")
axes[0].set_ylabel("Quantity")

sns.boxplot(y=sales_df.loc[sales_df["UnitPrice"] <= p99, "UnitPrice"], ax=axes[1])
axes[1].set_title("Unit price boxplot (up to 99th percentile)")
axes[1].set_ylabel("Unit price (£)")

plt.tight_layout()
plt.show()

# IQR-based counts provide a reproducible outlier flag, not an automatic deletion rule.
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum()), lower, upper

for column in ["Quantity", "UnitPrice", "Revenue"]:
    count, lower, upper = iqr_outlier_count(sales_df[column])
    print(f"{column}: {count:,} IQR-flagged values; lower={lower:.2f}, upper={upper:.2f}")


### 7.3 Monthly revenue trend

In [ ]:
monthly_revenue = (
    sales_df.groupby("YearMonth", as_index=False)["Revenue"]
    .sum()
    .sort_values("YearMonth")
)

plt.figure(figsize=(13, 5))
sns.lineplot(data=monthly_revenue, x="YearMonth", y="Revenue", marker="o")
plt.title("Monthly Positive-Sales Revenue")
plt.xlabel("Year and month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly_revenue)


### 7.4 Revenue by country

In [ ]:
country_revenue = (
    sales_df.groupby("Country", as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
)

top_countries = country_revenue.head(10).sort_values("Revenue")
plt.figure(figsize=(11, 6))
sns.barplot(data=top_countries, x="Revenue", y="Country")
plt.title("Top 10 Countries by Positive-Sales Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

display(country_revenue.head(10))


### 7.5 Top products by revenue

In [ ]:
# Exclude blank descriptions from the product-description chart.
product_revenue = (
    sales_df.dropna(subset=["Description"])
    .groupby("Description", as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
)

top_products = product_revenue.head(10).sort_values("Revenue")
plt.figure(figsize=(12, 7))
sns.barplot(data=top_products, x="Revenue", y="Description")
plt.title("Top 10 Product Descriptions by Positive-Sales Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

display(product_revenue.tail(10).sort_values("Revenue", ascending=False))


### 7.6 Correlation between numerical variables

In [ ]:
numeric_corr = sales_df[["Quantity", "UnitPrice", "Revenue"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(numeric_corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap: Quantity, Unit Price, and Revenue")
plt.tight_layout()
plt.show()

display(numeric_corr)


## 8. Findings and implications for future modeling

Use the tables and charts generated above to write a short interpretation in your own words. Address these points:

- **Data quality:** CustomerID is missing for a substantial number of records. Do not invent customer IDs; customer-level modeling should use records with known IDs and report this limitation.
- **Returns and cancellations:** Negative quantities and cancellation invoices are meaningful business events. Keep them available for separate analysis, while excluding them from positive-sales models when the target is ordinary sales revenue.
- **Skew and outliers:** Quantity, unit price, and revenue can contain extreme values. Inspect the business meaning of extremes rather than deleting them automatically. Consider robust scaling, log transformations, or tree-based models in later deliverables.
- **Time effects:** Monthly revenue and calendar features may help explain seasonality. Use time-aware validation where appropriate to avoid training on future data and testing on the past.
- **Country imbalance:** The United Kingdom dominates the records. Be careful when interpreting country comparisons or training models that use country as a predictor.
- **Association rules:** The transaction structure supports market-basket analysis. For Apriori/FP-Growth in Deliverable 3, group product codes by invoice and decide how cancellations/returns will be excluded.

### Suggested conclusion

The UCI Online Retail dataset is suitable for studying transaction-level sales and customer purchasing behavior. The initial inspection shows that data preparation is essential because customer identifiers are missing in many rows, exact duplicates exist, and the dataset includes returns, cancellations, and extreme numerical values. Separating all transactions from positive sales preserves the business meaning of returns while allowing a clear sales-focused analysis. The EDA results provide a basis for selecting features and defining targets in the next deliverable. Final conclusions should be updated using the actual tables and plots produced when this notebook is run.


## 9. Challenges and how they were addressed

| Challenge | Approach |
|---|---|
| Missing CustomerID values | Retained for general sales analysis; excluded only from customer-level analysis |
| Missing product descriptions | Retained where possible for transaction totals; excluded from description-based product labels |
| Duplicate records | Removed exact duplicate rows from the analysis copy |
| Negative quantities/cancellations | Preserved in the all-transactions dataset and excluded from positive-sales analysis |
| Extreme values | Flagged with IQR and displayed with capped chart ranges; not automatically deleted |
| Strong country imbalance | Reported country counts/revenue and noted the limitation in interpretation |

## 10. References

Chen, D. (2015). *Online retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

UCI Machine Learning Repository. (n.d.). *Online Retail*. https://archive.ics.uci.edu/dataset/352/online+retail
